# Setup — Download the L5PC example inputs

This notebook fetches the inputs needed by the four `_emodel_optimization` pipeline stages
from the BluePyEModel L5PC example
([README.rst](https://github.com/openbraininstitute/BluePyEModel/blob/main/examples/L5PC/README.rst)).

**What gets downloaded** (into `obi-output/L_emodel_optimization_inputs/`):

| Path | Source |
|---|---|
| `ephys_data/C060109A1-SR-C1/` | https://github.com/BlueBrain/SSCxEModelExamples/tree/main/feature_extraction/input-traces/C060109A1-SR-C1 (the same data fetched by `download_ephys_data.sh`) |
| `morphologies/C060114A5.asc` | BluePyEModel `examples/L5PC/morphologies/` |
| `mechanisms/*.mod` | BluePyEModel `examples/L5PC/mechanisms/` |
| `config/params/pyr.json` | BluePyEModel `examples/L5PC/config/params/pyr.json` |
| `config/recipes.json` | BluePyEModel `examples/L5PC/config/recipes.json` |

The pipeline notebooks (`01_efeature_extraction.ipynb` … `04_export_final_model.ipynb`)
read these inputs directly from this folder and only run if `bluepyemodel` is installed
in the active venv.


## Install BluePyEModel

In [ ]:
import subprocess
import sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "bluepyemodel"],
    check=True,
)


## Resolve paths

In [ ]:
from pathlib import Path

inputs_root = (Path.cwd() / "../../../obi-output/L_emodel_optimization_inputs").resolve()
inputs_root.mkdir(parents=True, exist_ok=True)
print("Inputs root:", inputs_root)


## Download the experimental ephys traces

This is what `download_ephys_data.sh` does in BluePyEModel — fetch every file under
`feature_extraction/input-traces/C060109A1-SR-C1/` from the SSCx example repo into
`./ephys_data/C060109A1-SR-C1/`. We re-implement it here in pure Python so the notebook
works on every platform.


In [ ]:
import json
import urllib.request

USER = "BlueBrain"
REPO = "SSCxEModelExamples"
BRANCH = "main"
FOLDER = "feature_extraction/input-traces/C060109A1-SR-C1"
ephys_dir = inputs_root / "ephys_data" / "C060109A1-SR-C1"
ephys_dir.mkdir(parents=True, exist_ok=True)

api_url = (
    f"https://api.github.com/repos/{USER}/{REPO}/git/trees/{BRANCH}?recursive=1"
)
with urllib.request.urlopen(api_url) as resp:
    tree = json.load(resp)["tree"]

paths = [item["path"] for item in tree if item["path"].startswith(FOLDER + "/")]
print(f"Found {len(paths)} files under {FOLDER}/")

for relpath in paths:
    fname = Path(relpath).name
    target = ephys_dir / fname
    if target.exists():
        continue
    raw_url = f"https://raw.githubusercontent.com/{USER}/{REPO}/{BRANCH}/{relpath}"
    print("Downloading", fname)
    with urllib.request.urlopen(raw_url) as resp, target.open("wb") as fh:
        fh.write(resp.read())

print("Total ibw files in", ephys_dir, ":", len(list(ephys_dir.glob("*.ibw"))))


## Download the morphology, mechanisms, params, and recipes

These are pulled from the BluePyEModel repo's `examples/L5PC/` directory. We reuse the
recipe verbatim and overwrite its `pipeline_settings` from the OBI-ONE blocks at run time.


In [ ]:
BPE_USER = "openbraininstitute"
BPE_REPO = "BluePyEModel"
BPE_BRANCH = "main"
BPE_BASE = (
    f"https://raw.githubusercontent.com/{BPE_USER}/{BPE_REPO}/{BPE_BRANCH}/examples/L5PC"
)


def fetch(remote: str, local: Path) -> None:
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.exists():
        print("  cached", local.relative_to(inputs_root))
        return
    with urllib.request.urlopen(f"{BPE_BASE}/{remote}") as resp, local.open("wb") as fh:
        fh.write(resp.read())
    print("  fetched", local.relative_to(inputs_root))


fetch("morphologies/C060114A5.asc", inputs_root / "morphologies" / "C060114A5.asc")
fetch("config/params/pyr.json", inputs_root / "config" / "params" / "pyr.json")
fetch("config/recipes.json", inputs_root / "config" / "recipes.json")

# Pull every .mod file out of the mechanisms tree.
mech_api = (
    f"https://api.github.com/repos/{BPE_USER}/{BPE_REPO}/contents/examples/L5PC/mechanisms"
    f"?ref={BPE_BRANCH}"
)
with urllib.request.urlopen(mech_api) as resp:
    mech_entries = json.load(resp)
for entry in mech_entries:
    if entry["type"] == "file" and entry["name"].endswith(".mod"):
        fetch(f"mechanisms/{entry['name']}", inputs_root / "mechanisms" / entry["name"])

print("\nInputs ready:", inputs_root)


## Inspect the recipes file

The recipe ships with `optimiser='MO-CMA'` and `max_ngen=100`. Stage 01 will overwrite
those values from the `OptimizationSettings` / `OptimizationParams` blocks at run time
(the bundled example sets `optimiser='CMA_ES'`, `max_ngen=2`, `offspring_size=4` so the
demo finishes in minutes rather than hours).


In [ ]:
import json

recipes = json.loads((inputs_root / "config" / "recipes.json").read_text())
print(json.dumps(recipes["L5PC"]["pipeline_settings"], indent=2))
